# Limited translation of EMGSD
Machine translate 20% of MGSD
Translate the sentences of SeeGULL and WinoQueer that the EMGSD uses

In [1]:
!python3 --version
!conda install -c conda-forge sentencepiece -y
!pip install torch

!pip install -U transformers
!pip install datasets
!pip install sacremoses

Python 3.10.19
Channels:
 - conda-forge
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.9.1
    latest version: 25.11.0

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /home/ec2-user/anaconda3/envs/python3

  added / updated specs:
    - sentencepiece


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    libsentencepiece-0.2.1     |       h1d72405_1         838 KB  conda-forge
    sentencepiece-0.2.1        |       hc5512b6_1          19 KB  conda-forge
    sentencepiece-python-0.2.1 |  py310h1469a80_1         3.0 MB  conda-forge
    sentencepiece-spm-0.2.1    |       h1d72405_1          85 KB  conda-forge
    ------------------------------------------------------------
                                           Total:         3.9 MB

The fo

In [2]:
import pandas as pd
import torch
import gc
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
from datasets import Dataset
from tqdm.auto import tqdm

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


In [3]:
FILE_MGSD = "MGSD_EN.csv"
FILE_SEAGULL = "SeeGULLGPTAugmentation_EN.csv"
FILE_WINO = "WinoqueerGPTAugmentation_EN.csv"
OUTPUT_FILE = "Dutch_EMGSD_Combined100.csv"

In [4]:
# GPU Setup
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if device == 0 else 'CPU'})")

Using device: 0 (Tesla T4)


In [5]:
# Initialize Translator (En -> Nl)
model_ckpt = "Helsinki-NLP/opus-mt-en-nl"
translator = pipeline("translation", model=model_ckpt, device=device)

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/316M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/316M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/790k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/814k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [6]:
def translate_dataset(df, name, batch_size=128):
    """
    Translates 'text' column of a dataframe.
    Returns the dataframe with new '_nl' columns.
    """
    print(f"\n Processing {name} ({len(df)} rows)...")
    
    # Convert to Hugging Face Dataset for speed, makesure columns are strings to prevent errors
    df['text'] = df['text'].fillna("").astype(str)
    if 'text_with_marker' in df.columns:
        df['text_with_marker'] = df['text_with_marker'].fillna("").astype(str)
    
    hf_ds = Dataset.from_pandas(df)
    
    # Translate 'text'
    print(f"  - Translating 'text' column...")
    results_text = []
    for out in tqdm(translator(KeyDataset(hf_ds, "text"), batch_size=batch_size, truncation=True), total=len(hf_ds)):
        results_text.append(out[0]['translation_text'])

    df['text_nl'] = results_text
    
    # Clean GPU memory between datasets
    torch.cuda.empty_cache()
    gc.collect()
    
    return df


In [7]:
# LOAD & SAMPLE MGSD
print("\n--- STEP 1: LOAD & DON'T SAMPLE MGSD ---")
df_mgsd = pd.read_csv(FILE_MGSD)
print(f"Original MGSD Size: {len(df_mgsd)}")

# Sample 20% randomly (random_state for reproducibility)
# df_mgsd_sampled = df_mgsd.sample(frac=0.2, random_state=42).copy()
df_mgsd_sampled = df_mgsd.copy()
print(f"Sampled MGSD Size (20%): {len(df_mgsd_sampled)}")

# Translate MGSD Sample
df_mgsd_dutch = translate_dataset(df_mgsd_sampled, "MGSD_Sampled")




--- STEP 1: LOAD & DON'T SAMPLE MGSD ---
Original MGSD Size: 51867
Sampled MGSD Size (20%): 51867

 Processing MGSD_Sampled (51867 rows)...
  - Translating 'text' column...


  0%|          | 0/51867 [00:00<?, ?it/s]


--- STEP 2: PROCESS SEAGULL & WINOQUEER ---

 Processing SeeGULL_Full (2070 rows)...
  - Translating 'text' column...


  0%|          | 0/2070 [00:00<?, ?it/s]

In [9]:
# Load WinoQueer
df_wino = pd.read_csv(FILE_WINO)
df_wino_dutch = translate_dataset(df_wino, "WinoQueer_Full")




 Processing WinoQueer_Full (3264 rows)...
  - Translating 'text' column...


  0%|          | 0/3264 [00:00<?, ?it/s]

In [10]:
# ==========================================
# STEP 3: MERGE & SAVE
print("\n--- STEP 3: MERGE & SAVE ---")

# source col add
df_mgsd_dutch['source_dataset'] = 'MGSD_20pct'
df_seagull_dutch['source_dataset'] = 'SeeGULL'
df_wino_dutch['source_dataset'] = 'WinoQueer'

# Concatenate all three
final_df = pd.concat([df_mgsd_dutch, df_seagull_dutch, df_wino_dutch], ignore_index=True)

final_df.to_csv(OUTPUT_FILE, index=False)

print(f"✅ SUCCESS! Created {OUTPUT_FILE}")
print(f"Total Rows: {len(final_df)}")
print("Columns:", final_df.columns.tolist())


--- STEP 3: MERGE & SAVE ---
✅ SUCCESS! Created Dutch_EMGSD_Combined100.csv
Total Rows: 57201
Columns: ['group', 'text', 'text_with_marker', 'category', 'data_source', 'multi-label', 'split', 'label', 'text_nl', 'source_dataset', 'phrase']
